# A New Theory of Magnetic Storms (Chapman & Ferraro, 1931): Implementation / 구현

이 노트북은 Chapman-Ferraro 이론의 핵심 개념을 구현합니다:
1. 하전 입자의 자이로 운동과 플라즈마 차폐 / Gyro motion and plasma shielding
2. 전도판의 image dipole 효과 / Image dipole effect of conducting sheet
3. 플라즈마 전면의 감속 계산 / Retardation of the plasma front
4. 자기권계면(magnetopause) 위치의 압력 균형 / Pressure balance at the magnetopause
5. Sudden commencement의 자기장 변화 시뮬레이션 / SC magnetic field simulation
6. Chapman-Ferraro의 원래 수치 결과 재현 / Reproducing original numerical results

This notebook implements the core concepts of the Chapman-Ferraro theory of magnetic storms,
focusing on the physics of plasma-field interaction, cavity formation, and the magnetopause.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from scipy.integrate import solve_ivp

plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

## Part 1: 하전 입자의 자이로 운동 / Charged Particle Gyration

Chapman-Ferraro 이론의 출발점: 자기장 속 하전 입자는 **자이로 운동(gyration)**을 합니다.
Larmor 반경 $r_L = mv_\perp / (|q|B)$이 자기장 변화 스케일보다 훨씬 작으면,
플라즈마는 **집단적으로** 자기장 침투를 막습니다.

Starting point of Chapman-Ferraro: charged particles gyrate in magnetic fields.
When the Larmor radius $r_L$ is much smaller than the field variation scale,
the plasma collectively prevents field penetration.

$$r_L = \frac{mv_\perp}{|q|B}$$

In [ ]:
def simulate_gyration(
    q: float, m: float, B: float, v_perp: float, v_parallel: float,
    t_max: float, n_steps: int = 5000
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Simulate charged particle gyration in a uniform magnetic field B along z.

    Args:
        q: Charge in Coulombs.
        m: Mass in kg.
        B: Magnetic field strength in Tesla.
        v_perp: Perpendicular velocity in m/s.
        v_parallel: Parallel velocity in m/s.
        t_max: Maximum simulation time in seconds.
        n_steps: Number of time steps.

    Returns:
        Tuple of (t, x, y, z) arrays.
    """
    omega_c = abs(q) * B / m  # cyclotron frequency
    r_L = m * v_perp / (abs(q) * B)  # Larmor radius
    sign = np.sign(q)

    t = np.linspace(0, t_max, n_steps)
    x = r_L * np.sin(sign * omega_c * t)
    y = r_L * (1 - np.cos(sign * omega_c * t))
    z = v_parallel * t

    return t, x, y, z


# Physical constants
e = 1.602e-19   # C
m_p = 1.673e-27  # kg (proton)
m_e = 9.109e-31  # kg (electron)
R_E = 6.371e6    # m (Earth radius)

# Solar wind conditions (Chapman-Ferraro assumed ~1000 km/s, T~6000K)
v_sw = 1e6       # m/s (1000 km/s, as in the paper)
T = 6000          # K
k_B = 1.381e-23
v_th_p = np.sqrt(k_B * T / m_p)  # thermal velocity for proton
v_th_e = np.sqrt(k_B * T / m_e)

B_surface = 5e-5  # T (Earth's equatorial surface field ~0.5 Gauss)
B_10RE = B_surface * (1/10)**3  # field at 10 R_E

r_L_proton = m_p * v_th_p / (e * B_surface)
r_L_electron = m_e * v_th_e / (e * B_surface)
r_L_proton_10RE = m_p * v_th_p / (e * B_10RE)

print("=== Larmor Radii / 자이로 반경 ===")
print(f"Proton at surface (B={B_surface*1e6:.0f} μT): r_L = {r_L_proton:.1f} m")
print(f"Electron at surface: r_L = {r_L_electron:.3f} m")
print(f"Proton at 10 R_E (B={B_10RE*1e9:.1f} nT): r_L = {r_L_proton_10RE:.0f} m")
print(f"\nEarth radius: {R_E:.2e} m")
print(f"r_L / R_E (surface): {r_L_proton / R_E:.2e}")
print(f"r_L / R_E (10 R_E):  {r_L_proton_10RE / R_E:.2e}")
print("\n→ Larmor radius << field scale → collective shielding possible")
print("  자이로 반경 << 자기장 스케일 → 집단적 차폐 가능")

In [ ]:
# Visualize proton vs electron gyration
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Proton
omega_p = e * B_surface / m_p
T_p = 2 * np.pi / omega_p
t_p, x_p, y_p, z_p = simulate_gyration(e, m_p, B_surface, v_th_p, 0, 3*T_p)

axes[0].plot(x_p, y_p, 'b-', lw=1.5)
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('y (m)')
axes[0].set_title(f'Proton Gyration / 양성자 자이로 운동\n'
                   f'$r_L$ = {r_L_proton:.1f} m, $f_c$ = {omega_p/(2*np.pi):.0f} Hz')
axes[0].set_aspect('equal')
axes[0].plot(0, 0, 'r+', ms=10, mew=2)

# Electron
omega_e = e * B_surface / m_e
T_e = 2 * np.pi / omega_e
t_e, x_e, y_e, z_e = simulate_gyration(-e, m_e, B_surface, v_th_e, 0, 3*T_e)

axes[1].plot(x_e, y_e, 'r-', lw=1.5)
axes[1].set_xlabel('x (m)')
axes[1].set_ylabel('y (m)')
axes[1].set_title(f'Electron Gyration / 전자 자이로 운동\n'
                   f'$r_L$ = {r_L_electron:.4f} m, $f_c$ = {omega_e/(2*np.pi):.0f} Hz')
axes[1].set_aspect('equal')
axes[1].plot(0, 0, 'r+', ms=10, mew=2)

fig.suptitle('Chapman-Ferraro §2: Charged Particle Gyration in Uniform B Field',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Part 2: 쌍극자 자기장과 Image Dipole / Dipole Field and Image Dipole

Chapman-Ferraro의 핵심 모델: 전도성 평면이 자기 쌍극자에 접근할 때, **image method**를 사용합니다.
완전 전도판은 쌍극자의 **거울 상(image)**을 만들어, 판 앞쪽의 자기장을 **두 배**로 증가시킵니다.

The key model: when a conducting sheet approaches a magnetic dipole,
the image method doubles the field on the dipole side.

$$\vec{B}_{\text{total}} = \vec{B}_{\text{dipole}} + \vec{B}_{\text{image}} \approx 2\vec{B}_{\text{dipole}} \text{ (at the sheet)}$$

In [ ]:
def dipole_field(x: np.ndarray, z: np.ndarray, M: float) -> tuple[np.ndarray, np.ndarray]:
    """Calculate magnetic dipole field in the x-z plane (dipole along z).

    Args:
        x: x-coordinates (in Earth radii).
        z: z-coordinates (in Earth radii).
        M: Magnetic moment (normalized, set to 1).

    Returns:
        Tuple of (Bx, Bz) field components.
    """
    r = np.sqrt(x**2 + z**2)
    r = np.where(r < 0.5, np.nan, r)  # avoid singularity near origin
    r5 = r**5
    Bx = 3 * M * x * z / r5
    Bz = M * (3 * z**2 - r**2) / r5
    return Bx, Bz


def dipole_with_image(
    x: np.ndarray, z: np.ndarray, M: float, d: float
) -> tuple[np.ndarray, np.ndarray]:
    """Calculate dipole + image dipole field.

    The image dipole is placed at z = 2*d (mirror of origin across z=d plane).
    The image has opposite moment direction to create the conducting boundary.

    Args:
        x: x-coordinates.
        z: z-coordinates.
        M: Magnetic moment.
        d: Distance of conducting sheet from origin (in R_E).

    Returns:
        Tuple of (Bx, Bz) total field components.
    """
    # Original dipole at origin
    Bx1, Bz1 = dipole_field(x, z, M)
    # Image dipole at z = 2d with same moment direction
    Bx2, Bz2 = dipole_field(x, z - 2*d, M)
    # Mask the region beyond the conducting sheet (z > d)
    mask = z > d
    Bx_total = np.where(mask, 0, Bx1 + Bx2)
    Bz_total = np.where(mask, 0, Bz1 + Bz2)
    return Bx_total, Bz_total


# Plot dipole field with and without conducting sheet
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

x = np.linspace(-8, 8, 200)
z = np.linspace(-8, 12, 250)
X, Z = np.meshgrid(x, z)

# Pure dipole
Bx, Bz = dipole_field(X, Z, 1.0)
B_mag = np.sqrt(Bx**2 + Bz**2)
B_mag_log = np.log10(np.clip(B_mag, 1e-6, None))

ax = axes[0]
ax.streamplot(X, Z, Bx, Bz, color=B_mag_log, cmap='viridis',
              density=1.5, linewidth=0.8, arrowsize=0.8)
earth = Circle((0, 0), 1, color='steelblue', zorder=5)
ax.add_patch(earth)
ax.set_xlim(-8, 8)
ax.set_ylim(-8, 12)
ax.set_xlabel('x ($R_E$)')
ax.set_ylabel('z ($R_E$) — Sun direction')
ax.set_title('Pure Dipole Field / 순수 쌍극자장')
ax.set_aspect('equal')

# Dipole + image (conducting sheet at z = 8 R_E)
d_sheet = 8.0
Bx_img, Bz_img = dipole_with_image(X, Z, 1.0, d_sheet)
B_mag_img = np.sqrt(Bx_img**2 + Bz_img**2)
B_mag_img_log = np.log10(np.clip(B_mag_img, 1e-6, None))

ax = axes[1]
ax.streamplot(X, Z, Bx_img, Bz_img, color=B_mag_img_log, cmap='viridis',
              density=1.5, linewidth=0.8, arrowsize=0.8)
earth2 = Circle((0, 0), 1, color='steelblue', zorder=5)
ax.add_patch(earth2)
ax.axhline(y=d_sheet, color='red', lw=3, label=f'Conducting sheet at {d_sheet:.0f} $R_E$')
ax.fill_between(x, d_sheet, 12, alpha=0.2, color='red', label='Plasma (shielded)')
ax.set_xlim(-8, 8)
ax.set_ylim(-8, 12)
ax.set_xlabel('x ($R_E$)')
ax.set_ylabel('z ($R_E$) — Sun direction')
ax.set_title(f'Dipole + Image (Sheet at {d_sheet:.0f} $R_E$)\n'
             f'Chapman-Ferraro Cavity / 쌍극자 + 전도판')
ax.set_aspect('equal')
ax.legend(loc='lower right')

fig.suptitle('Chapman-Ferraro §7: Image Dipole Method', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 3: 자기장 압축과 Sudden Commencement / Field Compression and SC

전도판이 쌍극자에 접근함에 따라, 지표면 자기장이 증가합니다 (image dipole 효과).
이것이 자기 폭풍의 **sudden commencement (SC)**를 설명합니다.

As the conducting sheet approaches, the surface field increases due to the image dipole.
This explains the **sudden commencement** at storm onset.

$$\Delta B_{\text{surface}} = B_0 \cdot \left(\frac{R_E}{2d}\right)^3$$

여기서 $d$는 전도판까지의 거리, image dipole이 $2d$에 위치하므로 기여는 $(R_E/2d)^3$에 비례합니다.

In [ ]:
def sc_field_increase(d: np.ndarray, B0: float = 30000.0) -> np.ndarray:
    """Calculate surface field increase due to image dipole.

    The image dipole at distance 2d from Earth center adds a field
    proportional to (R_E / 2d)^3 at the surface.

    Args:
        d: Distance of magnetopause in Earth radii.
        B0: Surface equatorial field in nT (default 30000 nT).

    Returns:
        Field increase at surface in nT.
    """
    return B0 * (1.0 / (2.0 * d))**3


# Simulate magnetopause approach and SC signature
d_values = np.linspace(20, 6, 500)  # magnetopause approaching from 20 to 6 R_E
delta_B = sc_field_increase(d_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Field increase vs distance
ax = axes[0]
ax.semilogy(d_values, delta_B, 'b-', lw=2)
ax.axhline(y=30, color='r', ls='--', alpha=0.7, label='Typical SC: ~30 nT')
ax.set_xlabel('Magnetopause distance ($R_E$)')
ax.set_ylabel('$\\Delta B$ at surface (nT)')
ax.set_title('Surface Field Increase vs Magnetopause Distance\n'
             '자기권계면 거리에 따른 지표면 자기장 증가')
ax.legend()
ax.invert_xaxis()

# Simulated SC time profile
ax = axes[1]
# Simple model: magnetopause moves from 15 to 8 R_E in ~5 minutes
t_minutes = np.linspace(-2, 10, 500)
d_t = np.where(t_minutes < 0, 15.0,
               np.where(t_minutes < 5, 15 - 1.4 * t_minutes, 8.0))
delta_B_t = sc_field_increase(d_t)
B_total = 30000 + delta_B_t  # baseline + SC

ax.plot(t_minutes, B_total - 30000, 'b-', lw=2)
ax.axvline(x=0, color='r', ls='--', alpha=0.5, label='SC onset')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('$\\Delta H$ (nT)')
ax.set_title('Simulated Sudden Commencement\n'
             'SC 시뮬레이션 (자기장 수평 성분 변화)')
ax.legend()

plt.tight_layout()
plt.show()

## Part 4: 플라즈마 전면의 감속 — Chapman-Ferraro ODE / Plasma Front Retardation

논문 §7.8의 핵심 계산: 플라즈마 전면이 자기장에 의해 감속되는 과정을 ODE로 기술합니다.

The core calculation of §7.8: the plasma front is decelerated by the magnetic field.

질량 축적: $\frac{dm}{dt} = \rho v$

자기 반발력: $F = -\frac{H^2}{2\pi} = -\frac{H_0^2 a^6}{2\pi z^6}$ (CGS)

무차원 변환 후:
$$\frac{d\phi}{d\xi} = \frac{5}{6} \cdot \frac{\sigma(1+\phi) - \phi}{\xi \phi}$$

여기서 $\phi = v/v_0$, $\xi = tv_0/a$, $\sigma = H_0^2/(4\pi\rho v_0^2 a)$

In [ ]:
def chapman_ferraro_ode(
    t: float, y: np.ndarray, rho: float, v0: float, B0: float, R_E: float
) -> list[float]:
    """ODE system for Chapman-Ferraro plasma front retardation.

    State variables: y = [z, v, m_accumulated]
    where z is position (m), v is velocity (m/s), m is accumulated mass (kg/m^2).

    Args:
        t: Time in seconds.
        y: State vector [z, v, m_accumulated].
        rho: Plasma mass density (kg/m^3).
        v0: Initial plasma velocity (m/s).
        B0: Earth's equatorial surface field (Tesla).
        R_E: Earth radius (m).

    Returns:
        Derivatives [dz/dt, dv/dt, dm/dt].
    """
    z, v, m_acc = y
    mu_0 = 4 * np.pi * 1e-7

    if z < 1.5 * R_E or v < 100:  # stop near Earth or if nearly stopped
        return [0, 0, 0]

    # Dipole field at distance z (equatorial plane)
    B = B0 * (R_E / z)**3

    # Magnetic pressure force per unit area
    F = -B**2 / (2 * mu_0)

    # Mass accumulation rate (per unit area)
    dm_dt = rho * v

    # Momentum equation: d(m*v)/dt = F
    # => m*dv/dt + v*dm/dt = F
    # => dv/dt = (F - v*dm/dt) / m
    # But initially m_acc -> 0, so we need the combined form:
    # d(m*v)/dt = F, dm/dt = rho*v
    # Let p = m*v => dp/dt = F, dm/dt = rho*v = rho*p/m
    if m_acc < 1e-10:
        m_acc = rho * v * 0.01  # small initial mass

    dv_dt = (F - v * dm_dt) / m_acc

    return [-v, dv_dt, dm_dt]  # dz/dt = -v (approaching Earth)


def simulate_retardation(
    N: float, v0: float = 1e6, B0: float = 5e-5, z0_RE: float = 30.0
) -> dict:
    """Simulate the retardation of a plasma front approaching Earth.

    Args:
        N: Number density in particles/cm^3.
        v0: Initial velocity in m/s.
        B0: Surface equatorial field in Tesla.
        z0_RE: Initial distance in Earth radii.

    Returns:
        Dictionary with time, position, velocity arrays.
    """
    R_E_m = 6.371e6
    N_m3 = N * 1e6  # convert cm^-3 to m^-3
    rho = N_m3 * m_p  # mass density

    z0 = z0_RE * R_E_m
    m0 = rho * v0 * 0.1  # small initial accumulated mass

    def event_stop(t, y, *args):
        return y[1] - 50  # stop when v < 50 m/s
    event_stop.terminal = True

    sol = solve_ivp(
        chapman_ferraro_ode, [0, 600], [z0, v0, m0],
        args=(rho, v0, B0, R_E_m),
        method='RK45', max_step=0.1, events=event_stop,
        dense_output=True
    )

    return {
        't': sol.t,
        'z_RE': sol.y[0] / R_E_m,
        'v_ratio': sol.y[1] / v0,
        'v': sol.y[1],
        'm': sol.y[2]
    }


# Simulate for different densities (as in Chapman-Ferraro Table)
densities = [0.1, 1, 10, 100, 1000]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

results = {}
for N, c in zip(densities, colors):
    res = simulate_retardation(N)
    results[N] = res
    axes[0].plot(res['z_RE'], res['v_ratio'], '-', color=c, lw=2,
                 label=f'N = {N} cm⁻³')
    axes[1].plot(res['t'], res['z_RE'], '-', color=c, lw=2,
                 label=f'N = {N} cm⁻³')

axes[0].set_xlabel('Distance from Earth center ($R_E$)')
axes[0].set_ylabel('$v/v_0$')
axes[0].set_title('Velocity vs Distance (cf. Fig. 7)\n'
                   '속도 대 거리 (논문 Fig. 7 재현)')
axes[0].legend()
axes[0].invert_xaxis()
axes[0].set_xlim(20, 2)
axes[0].set_ylim(0, 1.05)

axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Distance ($R_E$)')
axes[1].set_title('Position vs Time (cf. Fig. 8)\n'
                   '위치 대 시간 (논문 Fig. 8 재현)')
axes[1].legend()
axes[1].set_ylim(2, 22)
# Add unretarded straight line
t_line = np.linspace(0, 200, 100)
z_line = 30 - (1e6 * t_line / 6.371e6)
axes[1].plot(t_line, z_line, 'k--', lw=1, alpha=0.5, label='Unretarded')
axes[1].legend()

fig.suptitle('Chapman-Ferraro §7.8: Retardation of Plasma Front / 플라즈마 전면 감속',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 5: 압력 균형과 자기권계면 위치 / Pressure Balance and Magnetopause Position

Chapman-Ferraro 이론의 현대적 표현: **자기권계면(magnetopause)**의 위치는
태양풍 동압과 지구 자기압의 **균형 조건**으로 결정됩니다.

Modern formulation: the magnetopause position is set by pressure balance.

$$\frac{B^2(r)}{2\mu_0} = \frac{1}{2}\rho v^2 \quad \Rightarrow \quad r_{\text{mp}} = R_E \left(\frac{B_0^2}{2\mu_0 \rho v^2}\right)^{1/6}$$

In [ ]:
def magnetopause_standoff(
    n: float, v: float, B0: float = 3.12e-5
) -> float:
    """Calculate magnetopause standoff distance from pressure balance.

    Args:
        n: Solar wind number density in cm^-3.
        v: Solar wind speed in km/s.
        B0: Equatorial surface field in Tesla (default ~31200 nT).

    Returns:
        Standoff distance in Earth radii.
    """
    mu_0 = 4 * np.pi * 1e-7
    n_m3 = n * 1e6
    v_ms = v * 1e3
    rho = n_m3 * m_p
    P_dyn = 0.5 * rho * v_ms**2
    # B at magnetopause: B = B0 * (R_E/r)^3
    # B^2/(2*mu_0) = P_dyn => B0^2 * (R_E/r)^6 / (2*mu_0) = P_dyn
    # (R_E/r)^6 = 2*mu_0*P_dyn / B0^2
    # r/R_E = (B0^2 / (2*mu_0*P_dyn))^(1/6)
    r_RE = (B0**2 / (2 * mu_0 * P_dyn))**(1.0/6.0)
    return r_RE


# Plot magnetopause position as function of solar wind parameters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# vs density (at fixed velocity)
n_range = np.logspace(-1, 3, 200)  # 0.1 to 1000 cm^-3
r_mp_400 = [magnetopause_standoff(n, 400) for n in n_range]
r_mp_1000 = [magnetopause_standoff(n, 1000) for n in n_range]  # Chapman's value

ax = axes[0]
ax.semilogx(n_range, r_mp_400, 'b-', lw=2, label='v = 400 km/s (modern)')
ax.semilogx(n_range, r_mp_1000, 'r--', lw=2, label='v = 1000 km/s (Chapman)')
ax.axhline(y=10, color='gray', ls=':', alpha=0.7, label='Observed ~10 $R_E$')
ax.axvline(x=5, color='green', ls=':', alpha=0.7, label='Typical n ~5 cm⁻³')
ax.set_xlabel('Number density $n$ (cm$^{-3}$)')
ax.set_ylabel('Magnetopause distance ($R_E$)')
ax.set_title('Magnetopause Standoff vs Density\n자기권계면 거리 대 밀도')
ax.legend(fontsize=10)
ax.set_ylim(2, 25)

# vs velocity (at fixed density)
v_range = np.linspace(200, 1500, 200)
r_mp_n5 = [magnetopause_standoff(5, v) for v in v_range]
r_mp_n50 = [magnetopause_standoff(50, v) for v in v_range]

ax = axes[1]
ax.plot(v_range, r_mp_n5, 'b-', lw=2, label='n = 5 cm⁻³ (quiet)')
ax.plot(v_range, r_mp_n50, 'r-', lw=2, label='n = 50 cm⁻³ (storm)')
ax.axhline(y=10, color='gray', ls=':', alpha=0.7, label='Typical ~10 $R_E$')
ax.axvline(x=400, color='green', ls=':', alpha=0.7, label='Typical v ~400 km/s')
ax.set_xlabel('Solar wind velocity (km/s)')
ax.set_ylabel('Magnetopause distance ($R_E$)')
ax.set_title('Magnetopause Standoff vs Velocity\n자기권계면 거리 대 속도')
ax.legend(fontsize=10)
ax.set_ylim(2, 18)

fig.suptitle('Chapman-Ferraro Pressure Balance / 압력 균형 조건',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print key values
print("=== Magnetopause Standoff Distance / 자기권계면 거리 ===")
print(f"Quiet conditions (n=5, v=400): {magnetopause_standoff(5, 400):.1f} R_E")
print(f"Storm conditions (n=20, v=600): {magnetopause_standoff(20, 600):.1f} R_E")
print(f"Extreme storm (n=50, v=1000): {magnetopause_standoff(50, 1000):.1f} R_E")
print(f"Chapman's estimate (n=1, v=1000): {magnetopause_standoff(1, 1000):.1f} R_E")

## Part 6: 자기권 공동(Cavity) 형태 시각화 / Magnetospheric Cavity Visualization

Chapman-Ferraro §6.5에서 설명한 **포물선 형태의 빈 공간(hollow cavity)**을 시각화합니다.
태양 방향의 자기장이 가장 강하므로 흐름이 가장 멀리서 멈추고,
측면으로 갈수록 자기장이 약해져 흐름이 지구에 더 가까이 접근합니다.

Visualization of the parabolic cavity described in §6.5.
The field is strongest along the Sun-Earth line, so the plasma stalls farthest there,
while at the flanks the weaker field allows closer approach.

In [ ]:
def magnetopause_shape(
    r0: float, theta: np.ndarray, alpha: float = 0.5
) -> np.ndarray:
    """Calculate magnetopause shape using Shue et al. (1997) empirical model.

    r = r0 * (2 / (1 + cos(theta)))^alpha

    This is a modern parameterization of the Chapman-Ferraro cavity concept.

    Args:
        r0: Standoff distance in Earth radii.
        theta: Angle from Sun-Earth line in radians.
        alpha: Flaring parameter (0.5 for parabolic).

    Returns:
        Radial distance of magnetopause in Earth radii.
    """
    return r0 * (2.0 / (1.0 + np.cos(theta)))**alpha


fig, ax = plt.subplots(figsize=(12, 8))

theta = np.linspace(-np.pi * 0.85, np.pi * 0.85, 500)

# Quiet magnetopause
r0_quiet = 10.0
r_quiet = magnetopause_shape(r0_quiet, theta, alpha=0.58)
x_q = r_quiet * np.cos(theta)
y_q = r_quiet * np.sin(theta)

# Storm magnetopause (compressed)
r0_storm = 6.6
r_storm = magnetopause_shape(r0_storm, theta, alpha=0.58)
x_s = r_storm * np.cos(theta)
y_s = r_storm * np.sin(theta)

ax.plot(x_q, y_q, 'b-', lw=2.5, label=f'Quiet ($r_0$ = {r0_quiet} $R_E$)')
ax.plot(x_s, y_s, 'r-', lw=2.5, label=f'Storm ($r_0$ = {r0_storm} $R_E$)')

# Earth
earth = Circle((0, 0), 1, color='steelblue', zorder=5)
ax.add_patch(earth)
ax.annotate('Earth', (0, 0), fontsize=10, ha='center', va='center',
            color='white', fontweight='bold', zorder=6)

# Dipole field lines inside cavity
for L in [2, 3, 4, 5, 7]:
    theta_fl = np.linspace(-np.pi/2 * 0.95, np.pi/2 * 0.95, 200)
    r_fl = L * np.cos(theta_fl)**2
    x_fl = r_fl * np.cos(theta_fl)
    y_fl = r_fl * np.sin(theta_fl)
    ax.plot(x_fl, y_fl, 'gray', lw=0.5, alpha=0.5)

# Solar wind arrows
for y_arr in np.arange(-15, 16, 3):
    ax.annotate('', xy=(18, y_arr), xytext=(22, y_arr),
                arrowprops=dict(arrowstyle='->', color='orange', lw=1.5))

ax.annotate('Solar Wind\n태양풍 →', xy=(20, -18), fontsize=12,
            color='orange', ha='center', fontweight='bold')
ax.annotate('← Sun / 태양', xy=(22, 0), fontsize=10, color='orange')

# Shade the plasma region
ax.fill_betweenx(np.linspace(-25, 25, 100), 25, 25,
                  alpha=0.05, color='orange')

ax.set_xlim(-20, 25)
ax.set_ylim(-20, 20)
ax.set_xlabel('x ($R_E$) — Sun direction →', fontsize=12)
ax.set_ylabel('y ($R_E$)', fontsize=12)
ax.set_title('Chapman-Ferraro Cavity / 자기권 공동\n'
             '"A hollow space, roughly parabolic in form" (§6.5)',
             fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.legend(fontsize=12, loc='lower left')

plt.tight_layout()
plt.show()

## Part 7: 감속 ODE의 무차원 분석 / Dimensionless Analysis of Retardation ODE

Chapman-Ferraro 논문 §7.9의 무차원 운동 방정식을 직접 풀어, 논문의 Figure 6을 재현합니다.

Reproduce Figure 6 from the paper by solving the dimensionless ODE:

$$\frac{d\phi}{d\xi} = \frac{5}{6} \cdot \frac{\sigma(1+\phi) - \phi}{\xi\phi}$$

In [ ]:
def chapman_ferraro_dimensionless(
    xi: float, phi: np.ndarray, sigma: float
) -> list[float]:
    """Dimensionless Chapman-Ferraro ODE (equation 45 in the paper).

    Args:
        xi: Dimensionless time.
        phi: Dimensionless velocity [phi].
        sigma: Dimensionless parameter (magnetic/dynamic pressure ratio).

    Returns:
        [dphi/dxi]
    """
    p = phi[0]
    if abs(p) < 1e-10 or abs(xi) < 1e-10:
        return [0]
    dphi = (5.0 / 6.0) * (sigma * (1 + p) - p) / (xi * p)
    return [dphi]


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Solve for different sigma values
sigma_values = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]
colors_s = plt.cm.viridis(np.linspace(0.1, 0.9, len(sigma_values)))

ax = axes[0]
for sigma, c in zip(sigma_values, colors_s):
    xi_span = (0.01, 50)
    phi0 = [1.0 - 1e-6]  # start near phi=1

    sol = solve_ivp(
        chapman_ferraro_dimensionless, xi_span, phi0,
        args=(sigma,), method='RK45', max_step=0.05,
        dense_output=True
    )

    ax.plot(sol.t, sol.y[0], '-', color=c, lw=2, label=f'$\\sigma$ = {sigma}')

ax.set_xlabel('$\\xi$ (dimensionless time)', fontsize=12)
ax.set_ylabel('$\\phi = v/v_0$ (dimensionless velocity)', fontsize=12)
ax.set_title('Solutions of Chapman-Ferraro ODE (eq. 45)\n무차원 운동 방정식의 해')
ax.legend(fontsize=10)
ax.set_ylim(-0.1, 1.1)
ax.set_xlim(0, 40)

# Phase portrait (phi vs xi) — recreate Fig. 6
ax = axes[1]
sigma_fixed = 0.1

for phi_init in [0.99, 0.95, 0.9, 0.8, 0.6, 0.4, 0.2]:
    sol = solve_ivp(
        chapman_ferraro_dimensionless, (0.01, 80), [phi_init],
        args=(sigma_fixed,), method='RK45', max_step=0.05
    )
    ax.plot(sol.t, sol.y[0], 'b-', lw=1, alpha=0.6)

# Parabola phi^2 + phi - xi = 0 (the touching curve from eq. 46)
xi_par = np.linspace(0, 40, 200)
phi_par = (-1 + np.sqrt(1 + 4*xi_par)) / 2
ax.plot(xi_par, phi_par, 'r--', lw=2, label='$\\phi^2 + \\phi = \\xi$ (envelope)')

ax.set_xlabel('$\\xi$', fontsize=12)
ax.set_ylabel('$\\phi$', fontsize=12)
ax.set_title(f'Phase Portrait ($\\sigma$ = {sigma_fixed}) — cf. Fig. 6\n'
             f'위상 초상 — 논문 Fig. 6 재현')
ax.legend(fontsize=11)
ax.set_ylim(-0.5, 3)
ax.set_xlim(0, 40)

fig.suptitle('Chapman-Ferraro Dimensionless ODE / 무차원 운동 방정식',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 8: 종합 — 자기 폭풍의 초기 위상 / Synthesis — The Initial Phase of a Magnetic Storm

Chapman-Ferraro 이론의 전체 그림을 하나의 시뮬레이션으로 종합합니다:
1. 태양풍 플라즈마가 지구에 접근
2. 자기압에 의해 감속
3. 자기권계면에서 정지 (압력 균형)
4. 유도 전류가 지표면 자기장을 증가시킴 (SC)

Synthesis of the full Chapman-Ferraro picture in one simulation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Simulation parameters
N_sw = 5.0      # cm^-3 (modern typical value)
v_sw_km = 400   # km/s
B0 = 3.12e-5    # T

# Expected standoff
r_mp_expected = magnetopause_standoff(N_sw, v_sw_km, B0)
print(f"Expected magnetopause: {r_mp_expected:.1f} R_E")

# Simulate retardation
res = simulate_retardation(N_sw, v0=v_sw_km*1e3, B0=B0, z0_RE=25.0)

# Panel 1: Position vs time
ax = axes[0, 0]
ax.plot(res['t']/60, res['z_RE'], 'b-', lw=2)
ax.axhline(y=r_mp_expected, color='r', ls='--',
           label=f'Pressure balance: {r_mp_expected:.1f} $R_E$')
ax.axhline(y=1, color='brown', ls='-', lw=2, label='Earth surface')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Distance ($R_E$)')
ax.set_title('Plasma Front Position / 플라즈마 전면 위치')
ax.legend()
ax.set_ylim(0, 26)

# Panel 2: Velocity vs time
ax = axes[0, 1]
ax.plot(res['t']/60, res['v']/1e3, 'r-', lw=2)
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Velocity (km/s)')
ax.set_title('Plasma Front Velocity / 플라즈마 전면 속도')

# Panel 3: Magnetic field at surface vs time (SC simulation)
ax = axes[1, 0]
delta_B_t = sc_field_increase(res['z_RE'])
ax.plot(res['t']/60, delta_B_t, 'g-', lw=2)
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('$\\Delta B$ at surface (nT)')
ax.set_title('Sudden Commencement Signal / SC 신호')

# Panel 4: Pressure balance
ax = axes[1, 1]
r_range = np.linspace(2, 20, 200)
mu_0 = 4 * np.pi * 1e-7
R_E_m = 6.371e6

B_r = B0 * (1.0 / r_range)**3
P_mag = B_r**2 / (2 * mu_0) * 1e9  # in nPa

rho_sw = N_sw * 1e6 * m_p
P_dyn = 0.5 * rho_sw * (v_sw_km * 1e3)**2 * 1e9  # in nPa

ax.semilogy(r_range, P_mag, 'b-', lw=2, label='$B^2/2\\mu_0$ (magnetic)')
ax.axhline(y=P_dyn, color='r', ls='-', lw=2, label=f'$\\rho v^2/2$ = {P_dyn:.2f} nPa')
ax.axvline(x=r_mp_expected, color='green', ls='--',
           label=f'$r_{{mp}}$ = {r_mp_expected:.1f} $R_E$')
ax.set_xlabel('Distance ($R_E$)')
ax.set_ylabel('Pressure (nPa)')
ax.set_title('Pressure Balance / 압력 균형')
ax.legend(fontsize=10)
ax.set_xlim(2, 20)

fig.suptitle(
    f'Chapman-Ferraro Theory: Complete Initial Phase\n'
    f'n = {N_sw} cm⁻³, v = {v_sw_km} km/s → $r_{{mp}}$ = {r_mp_expected:.1f} $R_E$',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | Chapman-Ferraro (1931) | Modern Understanding / 현대적 이해 |
|---|---|---|
| Solar emission / 태양 방출 | Corpuscular stream (간헐적 가스 덩어리) | Continuous solar wind + CME |
| Field exclusion / 자기장 차폐 | Conducting sheet model (전도판 모델) | MHD frozen-in flux |
| Cavity / 공동 | "Hollow space, roughly parabolic" | Magnetosphere (자기권) |
| Boundary / 경계 | Induced current sheet | Magnetopause (자기권계면) |
| SC mechanism / SC 메커니즘 | Image dipole compression | Magnetopause compression |
| Standoff distance / 정지 거리 | ODE for retardation | $r_{mp} = R_E(B_0^2/2\mu_0\rho v^2)^{1/6}$ |
| Main phase / 주 위상 | Not explained (후속 논문) | Ring current injection |

### 다음 논문과의 연결 / Connection to Next Papers

| Next Paper / 다음 논문 | Connection / 연결 |
|---|---|
| **Chapman & Bartels (1940)** — #3 | Dst, Kp 지수 체계화 → 자기 폭풍의 정량적 분류 |
| **Parker (1958)** — #4 | "간헐적 덩어리" → "연속적 태양풍" 전환 |
| **Dungey (1961)** — #6 | 닫힌 cavity → IMF 남향 시 열린 자기권 |